# Layout-optimized chunking

Reads every PDF in `data/raw/kvaliteket/` and splits each into chunks along its headings. Sections longer than the chunk size are split further by size, and tables stay in the section they belong to.

The PDFs are real HR documents and notebook outputs are saved in this file: only print counts and statistics here, never chunk text.

In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd().parent

from backend.ai.pdf_reader import read_all_pdfs
from backend.ai.chunking.fixed_chunking import chunk_by_fixed_size, DEFAULT_CHUNK_SIZE
from backend.ai.chunking.customized_chunking import chunk_by_layout

In [ ]:
documents = read_all_pdfs(REPO_ROOT / "data" / "raw" / "kvaliteket")

chunks = [
    chunk
    for document in documents
    for chunk in chunk_by_layout(document["words"], document["source_pdf"])
]

In [ ]:
import re
from statistics import mean, median, pstdev

words_per_chunk = [
    len(chunk["text"].split())
    for chunk in chunks
]
chunks_from_split_sections = [
    chunk
    for chunk in chunks
    if re.search(r"\(\d+/\d+\)$", chunk["heading"])
]
tiny_chunk_word_limit = 3

print(
    f"{len(documents)} documents -> {len(chunks)} chunks "
    f"(max chunk size {DEFAULT_CHUNK_SIZE})"
)

print(f"Total words: {sum(words_per_chunk)}")
print(f"Average chunks per document: {len(chunks) / len(documents):.1f}")
print(f"Chunks from sections that were split by size: {len(chunks_from_split_sections)}")
print(f"Chunks with at most {tiny_chunk_word_limit} words: {sum(words <= tiny_chunk_word_limit for words in words_per_chunk)}")

print("\nWords per chunk:")
print(f"  min:    {min(words_per_chunk)}")
print(f"  max:    {max(words_per_chunk)}")
print(f"  mean:   {mean(words_per_chunk):.1f}")
print(f"  median: {median(words_per_chunk):.1f}")
print(f"  std:    {pstdev(words_per_chunk):.1f}")

In [ ]:
import matplotlib.pyplot as plt

EMBEDDING_MODEL_WORD_LIMIT = 96

plt.hist(words_per_chunk, bins=30)
plt.axvline(EMBEDDING_MODEL_WORD_LIMIT, color="red", linestyle="--", label="embedding model limit")
plt.xlabel("words per chunk")
plt.ylabel("number of chunks")
plt.legend()
plt.show()